# Semana 6: Funciones de Ventana y Limpieza
## Notebook Conceptual (NB1) – Window Functions en pandas

### Propósito de la sesión
Utilizar **funciones de ventana** (window functions) y técnicas de limpieza para análisis avanzado de datos. Implementaremos en pandas operaciones como ranking, diferencias con LAG, y medias móviles, y compararemos con la sintaxis SQL equivalente.

### Objetivos de aprendizaje
*   Generar datos de ventas diarias para análisis temporal.
*   Implementar funciones de **ranking** (ROW_NUMBER, RANK, DENSE_RANK) en pandas.
*   Calcular valores de filas anteriores y posteriores usando `shift()` (equivalente a LAG/LEAD).
*   Obtener diferencias con el mes anterior y medias móviles.
*   Comparar la sintaxis de pandas con la sintaxis SQL equivalente.
*   Aplicar funciones de limpieza básicas (manejo de nulos, formatos).

## Configuración Inicial

Importamos las librerías necesarias: pandas, numpy, matplotlib.

In [ ]:
# Importamos librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Configuración de pandas para mostrar más filas
pd.set_option('display.max_rows', 20)
pd.set_option('display.width', 1000)

# Configuración de matplotlib
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Fijamos semilla para reproducibilidad
np.random.seed(42)

print("🎯 Librerías importadas correctamente.")

---
## 1. Generación de Datos de Ventas Diarias

Simularemos datos de ventas diarias para varias tiendas a lo largo de varios meses.

In [ ]:
# Crear rango de fechas
fechas = pd.date_range(start='2025-01-01', end='2025-03-31', freq='D')

# Generar datos para 3 tiendas
tiendas = ['Tienda A', 'Tienda B', 'Tienda C']

# Lista para almacenar DataFrames
dfs = []

for tienda in tiendas:
    # Tendencia base + estacionalidad + ruido
    tendencia = np.linspace(100, 150, len(fechas)) if tienda == 'Tienda A' else \
               np.linspace(80, 120, len(fechas)) if tienda == 'Tienda B' else \
               np.linspace(60, 90, len(fechas))
    estacionalidad = 20 * np.sin(2 * np.pi * np.arange(len(fechas)) / 7)  # ciclo semanal
    ruido = np.random.normal(0, 10, len(fechas))
    ventas = tendencia + estacionalidad + ruido
    ventas = np.maximum(ventas, 0)  # ventas no negativas
    
    df_temp = pd.DataFrame({
        'fecha': fechas,
        'tienda': tienda,
        'ventas': ventas.round(2)
    })
    dfs.append(df_temp)

df_ventas = pd.concat(dfs, ignore_index=True)

print("🔷 Datos de ventas generados:")
display(df_ventas.head(10))
print(f"\nShape: {df_ventas.shape}")

In [ ]:
# Visualización rápida
plt.figure(figsize=(14, 6))
for tienda in tiendas:
    df_tienda = df_ventas[df_ventas['tienda'] == tienda]
    plt.plot(df_tienda['fecha'], df_tienda['ventas'], label=tienda, alpha=0.7)
plt.xlabel('Fecha')
plt.ylabel('Ventas')
plt.title('Ventas diarias por tienda')
plt.legend()
plt.grid(True)
plt.show()

---
## 2. Funciones de Ranking en pandas

Las funciones de ranking asignan una posición a cada fila según un orden. En SQL, tenemos `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`. En pandas, usamos `rank()` con diferentes métodos.

### 2.1. ROW_NUMBER() - numeración única

En pandas, podemos usar `cumcount()` después de ordenar, o `rank(method='first')`.

In [ ]:
# Ordenamos por ventas descendente dentro de cada tienda
df_ventas_sorted = df_ventas.sort_values(['tienda', 'ventas'], ascending=[True, False])

# ROW_NUMBER() equivalente: asignar número secuencial por grupo
df_ventas_sorted['row_number'] = df_ventas_sorted.groupby('tienda').cumcount() + 1

print("🔷 ROW_NUMBER (orden por ventas descendente):")
display(df_ventas_sorted[['tienda', 'fecha', 'ventas', 'row_number']].head(10))

### 2.2. RANK() y DENSE_RANK()

En pandas, `rank(method='min')` equivale a RANK (deja huecos), y `rank(method='dense')` equivale a DENSE_RANK (sin huecos).

In [ ]:
# Creamos datos con empates para ilustrar
df_empates = pd.DataFrame({
    'categoria': ['A', 'A', 'A', 'B', 'B', 'B'],
    'valor': [100, 100, 80, 200, 190, 190]
})

# RANK() - method='min'
df_empates['rank_min'] = df_empates.groupby('categoria')['valor'].rank(method='min', ascending=False)

# DENSE_RANK() - method='dense'
df_empates['dense_rank'] = df_empates.groupby('categoria')['valor'].rank(method='dense', ascending=False)

print("🔷 Comparación RANK vs DENSE_RANK:")
display(df_empates)

### 2.3. SQL equivalente

```sql
SELECT 
    categoria,
    valor,
    RANK() OVER (PARTITION BY categoria ORDER BY valor DESC) as rank,
    DENSE_RANK() OVER (PARTITION BY categoria ORDER BY valor DESC) as dense_rank
FROM tabla;
```

---
## 3. Funciones de Valor: LAG y LEAD

En SQL, `LAG()` accede a filas anteriores y `LEAD()` a filas posteriores. En pandas, usamos `shift()`.

In [ ]:
# Para una tienda específica, ordenada por fecha
df_tienda_a = df_ventas[df_ventas['tienda'] == 'Tienda A'].sort_values('fecha').copy()

# LAG(ventas, 1) - ventas del día anterior
df_tienda_a['ventas_dia_anterior'] = df_tienda_a['ventas'].shift(1)

# LEAD(ventas, 1) - ventas del día siguiente
df_tienda_a['ventas_dia_siguiente'] = df_tienda_a['ventas'].shift(-1)

# Diferencia con el día anterior
df_tienda_a['diferencia_diaria'] = df_tienda_a['ventas'] - df_tienda_a['ventas_dia_anterior']

print("🔷 LAG y LEAD en pandas (shift):")
display(df_tienda_a[['fecha', 'ventas', 'ventas_dia_anterior', 'diferencia_diaria', 'ventas_dia_siguiente']].head(10))

### 3.1. SQL equivalente

```sql
SELECT 
    fecha,
    ventas,
    LAG(ventas, 1) OVER (ORDER BY fecha) as ventas_dia_anterior,
    ventas - LAG(ventas, 1) OVER (ORDER BY fecha) as diferencia_diaria,
    LEAD(ventas, 1) OVER (ORDER BY fecha) as ventas_dia_siguiente
FROM ventas_tienda_a
ORDER BY fecha;
```

---
## 4. Diferencias con Mes Anterior

Queremos comparar las ventas totales de cada mes con el mes anterior. Primero agregamos por mes.

In [ ]:
# Agregamos ventas por mes y tienda
df_mensual = df_ventas.groupby([pd.Grouper(key='fecha', freq='M'), 'tienda'])['ventas'].sum().reset_index()
df_mensual = df_mensual.sort_values(['tienda', 'fecha'])

# Calculamos ventas del mes anterior para cada tienda
df_mensual['ventas_mes_anterior'] = df_mensual.groupby('tienda')['ventas'].shift(1)

# Diferencia absoluta y porcentual
df_mensual['diferencia_abs'] = df_mensual['ventas'] - df_mensual['ventas_mes_anterior']
df_mensual['diferencia_pct'] = (df_mensual['diferencia_abs'] / df_mensual['ventas_mes_anterior'] * 100).round(2)

print("🔷 Ventas mensuales y diferencia con mes anterior:")
display(df_mensual)

### 4.1. SQL equivalente

```sql
WITH ventas_mensuales AS (
    SELECT 
        DATE_TRUNC('month', fecha) as mes,
        tienda,
        SUM(ventas) as total_ventas
    FROM ventas
    GROUP BY DATE_TRUNC('month', fecha), tienda
)
SELECT 
    mes,
    tienda,
    total_ventas,
    LAG(total_ventas) OVER (PARTITION BY tienda ORDER BY mes) as ventas_mes_anterior,
    total_ventas - LAG(total_ventas) OVER (PARTITION BY tienda ORDER BY mes) as diferencia_abs,
    (total_ventas - LAG(total_ventas) OVER (PARTITION BY tienda ORDER BY mes)) * 100.0 / LAG(total_ventas) OVER (PARTITION BY tienda ORDER BY mes) as diferencia_pct
FROM ventas_mensuales
ORDER BY tienda, mes;
```

---
## 5. Media Móvil (Moving Average)

Calculamos la media móvil de 7 días (ventana semanal) para cada tienda.

In [ ]:
# Aseguramos orden por fecha para cada tienda
df_ventas = df_ventas.sort_values(['tienda', 'fecha'])

# Media móvil de 7 días (ventana centrada)
df_ventas['media_movil_7'] = df_ventas.groupby('tienda')['ventas'].transform(
    lambda x: x.rolling(window=7, min_periods=1).mean()
)

print("🔷 Media móvil de 7 días:")
display(df_ventas.head(15))

In [ ]:
# Visualización de la media móvil para Tienda A
df_tienda_a = df_ventas[df_ventas['tienda'] == 'Tienda A'].copy()

plt.figure(figsize=(14, 6))
plt.plot(df_tienda_a['fecha'], df_tienda_a['ventas'], label='Ventas diarias', alpha=0.5)
plt.plot(df_tienda_a['fecha'], df_tienda_a['media_movil_7'], label='Media móvil 7 días', linewidth=2, color='red')
plt.xlabel('Fecha')
plt.ylabel('Ventas')
plt.title('Ventas diarias y media móvil (7 días) - Tienda A')
plt.legend()
plt.grid(True)
plt.show()

### 5.1. SQL equivalente

```sql
SELECT 
    fecha,
    tienda,
    ventas,
    AVG(ventas) OVER (PARTITION BY tienda ORDER BY fecha ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) as media_movil_7
FROM ventas
ORDER BY tienda, fecha;
```

---
## 6. Funciones de Limpieza Básicas

### 6.1. Manejo de valores nulos

Introducimos algunos nulos y los tratamos.

In [ ]:
# Creamos una copia con nulos
df_con_nulos = df_ventas.copy()
df_con_nulos.loc[10:15, 'ventas'] = np.nan

print("🔷 Datos con nulos:")
display(df_con_nulos.iloc[8:18])

# COALESCE en pandas: fillna
df_con_nulos['ventas_rellenadas'] = df_con_nulos['ventas'].fillna(0)

# NULLIF en pandas: donde valor == 0, poner NaN
df_con_nulos['ventas_nullif'] = df_con_nulos['ventas'].mask(df_con_nulos['ventas'] == 0)

print("\n🔷 Después de fillna(0):")
display(df_con_nulos.iloc[8:18][['fecha', 'ventas', 'ventas_rellenadas']])

### 6.2. Limpieza de texto (simulación)

Simulamos una columna con nombres de tienda inconsistentes.

In [ ]:
# Creamos datos sucios
df_sucio = pd.DataFrame({
    'tienda': [' Tienda A  ', 'TIENDA B', 'tienda c', 'Tienda A', 'Tienda  B'],
    'ventas': [100, 200, 150, 110, 190]
})

print("🔷 Datos sucios:")
display(df_sucio)

# Limpieza: eliminar espacios, capitalizar, normalizar
df_sucio['tienda_limpia'] = df_sucio['tienda'].str.strip().str.title()

# Reemplazar valores
df_sucio['tienda_limpia'] = df_sucio['tienda_limpia'].replace({
    'Tienda  B': 'Tienda B',
    'Tienda  A': 'Tienda A'
})

print("\n🔷 Datos limpios:")
display(df_sucio)

---
## 7. Comparación con Sintaxis SQL

A modo de resumen, aquí tienes una tabla comparativa entre pandas y SQL para operaciones de ventana:

| Operación | SQL | pandas |
|-----------|-----|--------|
| ROW_NUMBER | `ROW_NUMBER() OVER (PARTITION BY col ORDER BY col2)` | `df.groupby('col').cumcount() + 1` (previo orden) |
| RANK | `RANK() OVER (PARTITION BY col ORDER BY col2)` | `df.groupby('col')['val'].rank(method='min')` |
| DENSE_RANK | `DENSE_RANK() OVER (PARTITION BY col ORDER BY col2)` | `df.groupby('col')['val'].rank(method='dense')` |
| LAG | `LAG(val, n) OVER (ORDER BY col)` | `df['val'].shift(n)` |
| LEAD | `LEAD(val, n) OVER (ORDER BY col)` | `df['val'].shift(-n)` |
| Media móvil | `AVG(val) OVER (ORDER BY col ROWS BETWEEN n PRECEDING AND CURRENT ROW)` | `df['val'].rolling(n).mean()` |
| Suma acumulada | `SUM(val) OVER (ORDER BY col)` | `df['val'].cumsum()` |

---
## Ejercicios para el Estudiante

1.  **Top N por categoría:** Para los datos de ventas, encuentra los 3 días con mayores ventas para cada tienda (usa ranking).

2.  **Diferencia con el mismo día de la semana anterior:** Usando `shift(7)`, calcula la variación de ventas respecto al mismo día de la semana anterior.

3.  **Acumulado anual:** Calcula las ventas acumuladas por tienda a lo largo del tiempo.

4.  **Limpieza de fechas:** Si tuvieras fechas en formato 'DD/MM/YYYY' como texto, conviértelas a datetime. (Investiga `pd.to_datetime()`).

5.  **Reflexión:** ¿En qué situaciones del mundo real usarías funciones de ventana en lugar de simples agregaciones con GROUP BY?

---
## Conclusiones de la Sesión

*   Las **funciones de ventana** permiten realizar cálculos sobre conjuntos de filas relacionadas sin colapsar el resultado.
*   En pandas, implementamos **ranking** con `groupby` y `rank()`, y **LAG/LEAD** con `shift()`.
*   Calculamos **diferencias con el mes anterior** agregando primero por mes y luego usando shift.
*   Obtuvimos **medias móviles** con `rolling().mean()`.
*   Aplicamos **limpieza básica**: manejo de nulos con `fillna()` y normalización de texto con `str` methods.
*   Comparamos la sintaxis de pandas con SQL, observando que ambas siguen conceptos similares.
*   Estas técnicas son fundamentales en análisis de datos, ETL y preparación de features para machine learning.

¡Ahora dominas las funciones de ventana y limpieza en pandas!